# Data Cleaning — Systematic Transformation of a Messy Dataset
**Objective:** Take a deliberately messy dataset and systematically transform it into a clean,
analysis-ready dataset, documenting every decision made along the way.

**Dataset used:** *(fill in — recommended: Titanic dataset from Kaggle, a classic messy dataset
with missing Age/Cabin/Embarked values and inconsistent formatting)*

**Tech stack:** Python, pandas, numpy, Jupyter Notebook


## Step 0 — Imports & Setup

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)


## Step 1 — Load Dataset & Data Quality Report
Replace `'messy_data.csv'` with your actual filename. The code below assumes a Titanic-style
dataset but works generically for any dataset — it inspects whatever columns actually exist.


In [ ]:
# --- Load the data ---
df = pd.read_csv('messy_data.csv', encoding='utf-8-sig')  # utf-8-sig strips BOM characters if present
print("Shape:", df.shape)
df.head()


In [ ]:
# --- Keep an untouched copy for the before/after comparison later ---
df_original = df.copy()

df.info()


In [ ]:
# --- DATA QUALITY REPORT ---

# 1. Null counts per column
null_report = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().mean() * 100).round(2)
}).sort_values('null_count', ascending=False)

print("=== NULL VALUES PER COLUMN ===")
print(null_report[null_report['null_count'] > 0])

# 2. Duplicate rows
dupe_count = df.duplicated().sum()
print(f"\n=== DUPLICATE ROWS ===\nTotal exact duplicate rows: {dupe_count}")

# 3. Data type overview (flag potential issues)
print("\n=== DATA TYPES ===")
print(df.dtypes)

# 4. Value range anomalies for numeric columns (quick look at min/max)
num_cols = df.select_dtypes(include=np.number).columns.tolist()
print("\n=== NUMERIC COLUMN RANGES (check for impossible values, e.g. negative Age) ===")
print(df[num_cols].agg(['min', 'max']))


**Observation (fill in):** Summarize the data quality report — which columns have the most
missing values? Are there duplicate rows? Any obviously impossible values (e.g. negative age,
fare of 0, future dates)?


## Step 2 — Missing Data Handling
Choose and justify a strategy **per column**. The code below shows the classic Titanic-style
approach — adjust column names and strategies to match your actual dataset and its null report
above.

**General rules of thumb used here:**
- Numeric column with moderate missingness → **median imputation** (robust to outliers/skew)
- Categorical column with a small number of missing values → **mode imputation**
- Column with the vast majority of values missing → **drop the column** (too little signal to
  impute reliably)
- A handful of missing rows in an otherwise-complete, important column → **row deletion**


In [ ]:
# --- Example: Age (numeric, moderate missingness) -> median imputation ---
if 'Age' in df.columns:
    median_age = df['Age'].median()
    df['Age'] = df['Age'].fillna(median_age)
    print(f"Filled missing Age with median: {median_age}")

# --- Example: Embarked (categorical, very few missing) -> mode imputation ---
if 'Embarked' in df.columns:
    mode_embarked = df['Embarked'].mode()[0]
    df['Embarked'] = df['Embarked'].fillna(mode_embarked)
    print(f"Filled missing Embarked with mode: {mode_embarked}")

# --- Example: Cabin (majority missing) -> drop column, but keep a flag first ---
if 'Cabin' in df.columns:
    missing_pct = df['Cabin'].isnull().mean() * 100
    if missing_pct > 50:
        df['Has_Cabin_Info'] = df['Cabin'].notnull().astype(int)  # preserve signal before dropping
        df = df.drop(columns=['Cabin'])
        print(f"Dropped 'Cabin' column ({missing_pct:.1f}% missing); "
              f"preserved as binary 'Has_Cabin_Info' flag instead.")

# --- Generic fallback for any other columns with missing values ---
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print("\nColumns still containing nulls after the targeted steps above:")
print(remaining_nulls)


**Justification (fill in):** For each column you handled above, write one line explaining
*why* you chose that strategy, e.g.:
- *Age → median imputation, because Age is numeric with moderate missingness (~20%) and median
  is robust to the right-skew typically seen in age distributions.*
- *Embarked → mode imputation, because only 1-2 rows were missing out of hundreds — imputing
  with the most common port has negligible impact.*
- *Cabin → dropped the column but kept a binary "Has_Cabin_Info" flag, because over 70% of
  values were missing (too sparse to impute reliably), but whether cabin info exists at all may
  still carry predictive signal (often correlated with ticket class/fare).*

If your dataset has other columns with missing values not covered above, add a code cell +
justification for each one following the same pattern.


## Step 3 — Duplicate Removal

In [ ]:
rows_before = df.shape[0]
df = df.drop_duplicates()
rows_after = df.shape[0]
duplicates_removed = rows_before - rows_after

print(f"Rows before duplicate removal: {rows_before}")
print(f"Rows after duplicate removal: {rows_after}")
print(f"Duplicate rows removed: {duplicates_removed}")


**Observation (fill in):** How many duplicate rows were found and removed? If zero, note that
explicitly — it's still a meaningful finding (the dataset had no exact duplicates).


## Step 4 — Standardisation
Normalise inconsistent formatting: text casing/variants, and date formats.


In [ ]:
# --- Example: standardise a categorical column with inconsistent casing/variants ---
# e.g. "Male"/"male"/"M" -> "Male". Adjust the column name and mapping to your actual data.
if 'Sex' in df.columns:
    print("Unique values before standardisation:", df['Sex'].unique())
    sex_map = {
        'male': 'Male', 'Male': 'Male', 'M': 'Male', 'm': 'Male',
        'female': 'Female', 'Female': 'Female', 'F': 'Female', 'f': 'Female'
    }
    df['Sex'] = df['Sex'].astype(str).str.strip().replace(sex_map)
    print("Unique values after standardisation:", df['Sex'].unique())

# --- Generic text cleanup for object/string columns: strip whitespace, fix casing ---
text_cols = df.select_dtypes(include='object').columns.tolist()
for col in text_cols:
    if col not in ['Name', 'Ticket']:  # skip columns where casing genuinely matters
        df[col] = df[col].astype(str).str.strip()

print("\nText columns cleaned (whitespace stripped):", text_cols)


In [ ]:
# --- Standardise date columns to proper datetime ---
# Add any date column names present in your dataset to this list
date_columns = []  # e.g. ['Date', 'InvoiceDate'] -- Titanic has none by default

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f"Converted '{col}' to datetime. Dtype now: {df[col].dtype}")

if not date_columns:
    print("No date columns specified for this dataset — skip if not applicable.")


**Observation (fill in):** Which inconsistencies did you find and fix (casing variants,
whitespace, mixed date formats)? Show a before/after example if useful.


## Step 5 — Outlier Detection (IQR Method)
For each key numeric column, detect outliers using the IQR method, then decide per column
whether to cap, remove, or retain them — and document why.


In [ ]:
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    return outliers, lower, upper

numeric_cols_to_check = [c for c in df.select_dtypes(include=np.number).columns
                          if c not in ['PassengerId', 'Survived', 'Pclass']]  # exclude IDs/categorical-coded numbers

for col in numeric_cols_to_check:
    outliers, lower, upper = detect_outliers_iqr(df[col].dropna())
    print(f"{col}: {len(outliers)} outliers detected (valid range: {lower:.2f} to {upper:.2f})")


In [ ]:
# --- Example decision: cap Fare at the upper IQR bound instead of removing rows ---
# (Capping preserves sample size while limiting the influence of extreme values)
if 'Fare' in df.columns:
    outliers, lower, upper = detect_outliers_iqr(df['Fare'].dropna())
    before_max = df['Fare'].max()
    df['Fare'] = df['Fare'].clip(lower=max(lower, 0), upper=upper)
    print(f"Fare capped to range [{max(lower,0):.2f}, {upper:.2f}]. "
          f"Max value changed from {before_max:.2f} to {df['Fare'].max():.2f}")


**Decision & justification (fill in) — per numeric column:**
- *Fare → capped at the upper IQR bound, because a few extremely high fares likely reflect
  genuine first-class luxury tickets rather than data errors, so we retain the rows but limit
  their outsized influence on aggregate statistics like mean.*
- *Age → retained as-is (no capping), because ages up to ~80 are biologically plausible and not
  true anomalies, even if statistically flagged as outliers by IQR.*

Adjust the above based on what `numeric_cols_to_check` actually flagged in your dataset — some
columns may warrant removal instead of capping if the outlier clearly represents a data entry
error (e.g. a negative age, or a price of 0 for a real transaction).


## Step 6 — Data Type Correction
Ensure every column has the correct dtype: IDs as string (not numeric, since arithmetic on IDs
is meaningless), dates as datetime, monetary values as float, categorical flags as category/int.


In [ ]:
# --- Example type corrections (adjust column names to your dataset) ---
if 'PassengerId' in df.columns:
    df['PassengerId'] = df['PassengerId'].astype(str)

if 'Pclass' in df.columns:
    df['Pclass'] = df['Pclass'].astype('category')

if 'Sex' in df.columns:
    df['Sex'] = df['Sex'].astype('category')

if 'Fare' in df.columns:
    df['Fare'] = df['Fare'].astype(float)

print("Final dtypes:")
print(df.dtypes)


**Observation (fill in):** Which columns had incorrect dtypes originally, and what did you
correct them to? Why does the correct dtype matter for later analysis (e.g. IDs as strings
prevent accidental averaging of ID numbers)?


## Step 7 — Before vs. After Summary Table

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Row count',
        'Total null values',
        'Duplicate rows',
        'Number of columns',
        'Numeric columns with correct dtype (float/int)',
    ],
    'Before Cleaning': [
        df_original.shape[0],
        int(df_original.isnull().sum().sum()),
        int(df_original.duplicated().sum()),
        df_original.shape[1],
        len(df_original.select_dtypes(include=np.number).columns),
    ],
    'After Cleaning': [
        df.shape[0],
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum()),
        df.shape[1],
        len(df.select_dtypes(include=np.number).columns),
    ]
})
summary


**Observation (fill in):** Summarize the overall improvement — e.g. "Null values reduced from
X to Y, N duplicate rows removed, all columns now have appropriate dtypes." This table is the
single clearest proof of the cleaning work performed.


## Step 8 — Save the Cleaned Dataset

In [ ]:
df.to_csv('cleaned_data.csv', index=False)
print("Cleaned dataset saved as 'cleaned_data.csv'")
print("Final shape:", df.shape)


## Conclusion

**Summary of cleaning steps performed:**
1. *(fill in)*
2. *(fill in)*
3. *(fill in)*

**Why this matters:** A clean, well-typed, deduplicated dataset with documented, justified
handling of missing values and outliers is essential before any downstream analysis, modeling,
or reporting — undocumented cleaning decisions are a common source of irreproducible or
misleading results in real-world data work.
